# GPT from scratch

Building a character-level transformer on tinyshakespeare, step by step.
See `MATERIALS.md` for the reference links. We derive each piece ourselves.

**Roadmap:** data → tokenizer → tensor+split → batching → bigram baseline → attention → blocks → scale up.

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Reproducibility
torch.manual_seed(1337)

# Apple Silicon: use the MPS GPU if available, else CPU
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)

torch 2.12.1 | device: mps


## Step 1 — Load & inspect the data

Before modeling anything, *look at your data*. How big is it? What characters live in it? That character set becomes our entire vocabulary.

In [3]:
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print('length in characters:', len(text))
print('--- first 250 chars ---')
print(text[:250])

length in characters: 1115394
--- first 250 chars ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [4]:
# The vocabulary = the set of unique characters, sorted for a stable ordering
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(*chars, sep='')
print('vocab_size:', vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size: 65


## Step 2 — Tokenizer (your turn)

A tokenizer maps text ↔ integers. We're going **character-level**: each char gets an integer id. (GPT-2 uses sub-word BPE with ~50k tokens — a tradeoff between vocab size and sequence length. We stay simple.)

**Build two lookups and two functions:**
- `stoi`: char → int, `itos`: int → char
- `encode(s)`: string → list[int]
- `decode(l)`: list[int] → string

Sanity check: `decode(encode("hii there")) == "hii there"`.

In [5]:
# TODO (build it yourself — the whole point):
stoi = {c:n for n,c in enumerate(chars)}  # dict: char -> int
itos = {n:c for n,c in enumerate(chars)}  # dict: int -> char

def encode(s):
    return list(map(lambda x: stoi[x], s))

def decode(l):
    return ''.join(map(lambda x: itos[x], l))

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


## Step 3 — Tensor + train/val split (your turn)

Now that `encode` turns text into integer ids, we do two things:

**1. Encode the *entire* corpus into a single tensor.**
The model doesn't consume Python lists — it consumes `torch.Tensor`s. So we wrap
`encode(text)` in a tensor. Two things to get right:
- **dtype** — these numbers are *indices* into an embedding table, not measurements.
  Which integer dtype does PyTorch require for indexing / embedding lookups?
- **shape/dtype check** — always print `.shape` and `.dtype` right after you build a
  tensor. It's the cheapest bug-catcher you have.

**2. Split into train and validation sets.**
Why? So we can tell *learning* apart from *memorizing*. The model trains only on the
train split; we measure loss on the held-out val split it has never seen. If train loss
keeps dropping while val loss climbs, it's memorizing, not generalizing.

- Convention here: first **90%** train, last **10%** val.
- Note we take a **contiguous** slice, not a random shuffle — think about *why* that
  matters for a model that reads a continuous stream of text. (What would shuffling
  characters destroy?)

In [14]:
# TODO(human): encode the whole corpus into a tensor, then split 90/10.

# 1) Build the tensor of ALL the data.
#    - Which torch dtype for integer indices? (hint: not float)
# data = torch.tensor(...)
# print(data.shape, data.dtype)
# print(data[:100])          # sanity: do these ids match the start of `text`?
data = torch.tensor(encode(text), dtype=torch.int64)
print(data.shape, data.dtype)
print(data[:100])

# 2) Split: first 90% -> train_data, rest -> val_data.
#    - n = ? (how many chars go to train)
n = int(0.9*data.shape[0])
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])
1003854 111540


## Step 4 — Batching (your turn)

We never feed the model the whole 1M-character stream at once — too big, and pointless.
Instead we feed **many small chunks**. Two dimensions to understand:

### The *time* dimension — `block_size` (a.k.a. context length)
`block_size` is the **maximum number of previous characters** the model may look at when
predicting the next one. Say `block_size = 8`.

Key insight most people miss: a single chunk of length **`block_size + 1`** secretly
contains **`block_size` training examples**, because the model learns to predict at
*every* position along the chunk:

```
chunk = train_data[:block_size+1]   # 9 characters
    context (x)              -> target (y)
    [18]                     -> 47
    [18, 47]                 -> 56
    [18, 47, 56]             -> 57
    [18, 47, 56, 57]         -> 58
    ... up to length block_size
```

So `x` is a chunk and `y` is **that same chunk shifted right by one**. Position `t` in `y`
is the character that should follow `x[:t+1]`. That's *why* we grab `block_size + 1`
characters — you need the extra one to have a target for the last input. (Bonus: training
on contexts of length 1..block_size means the model still works when it has seen very
little — important at generation time, when it starts from almost nothing.)

### The *batch* dimension — `batch_size`
GPUs are parallel; processing one chunk at a time wastes them. So we stack `batch_size`
**independent** chunks into one tensor and process them together. Each chunk starts at a
**random offset** into the data (the Step 3 nuance: don't reorder characters, but *do*
randomize which chunks you sample).

### The shapes you're aiming for
Both `x` and `y` come out as **`(batch_size, block_size)`** — conventionally **`(B, T)`**
for Batch × Time. Burn `(B, T)` into memory; every tensor from here on is described by its
shape, and this is the first one.

**Your job:** (1) an illustration cell that unrolls ONE chunk into its context→target
pairs (so you *see* the shift), then (2) a `get_batch(split)` that returns a random
`(B, T)` batch of `x` and `y`.

In [19]:
# Hyperparameters (given — these are just config, not the lesson)
block_size = 8   # T: max context length
batch_size = 4   # B: independent sequences per batch

# --- Part 1: SEE the shift. Unroll one chunk into context -> target pairs. ---
# TODO(human):
x = train_data[:block_size]
y = train_data[1:block_size+1]         # y is x shifted right by one
for t in range(block_size):
    context = x[:t+1]
    target  = y[t]
    print(f"context {context.tolist()} -> target {target}")
# Look at the output: does each target equal the NEXT id after the context? Convince yourself.


# --- Part 2: get_batch. Return a random (B, T) batch of x and y. ---
# TODO(human):
def get_batch(split):
    data = train_data if split == 'train' else val_data
    # 1) pick batch_size random start offsets into `data`.
    #    - highest legal start so a full chunk of block_size+1 still fits? -> len(data) - block_size
    #    - which torch fn gives you `batch_size` random ints in [0, high)?  (hint: torch.randint)
    ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
    # 2) build x and y by stacking the chunks. Each row: data[i : i+block_size] for x,
    #    and the same window shifted by one for y. (hint: torch.stack a list comprehension)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    # 3) move both to `device` so they live where the model will.
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print('xb.shape', xb.shape, '| yb.shape', yb.shape)   # expect (batch_size, block_size)
print(xb)
print(yb)


context [18] -> target 47
context [18, 47] -> target 56
context [18, 47, 56] -> target 57
context [18, 47, 56, 57] -> target 58
context [18, 47, 56, 57, 58] -> target 1
context [18, 47, 56, 57, 58, 1] -> target 15
context [18, 47, 56, 57, 58, 1, 15] -> target 47
context [18, 47, 56, 57, 58, 1, 15, 47] -> target 58
xb.shape torch.Size([4, 8]) | yb.shape torch.Size([4, 8])
tensor([[57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46]], device='mps:0')
tensor([[43, 60, 43, 52,  1, 63, 43, 39],
        [43, 42,  8,  0, 25, 63,  1, 45],
        [42,  5, 57,  1, 57, 39, 49, 43],
        [57, 58, 63,  6,  1, 58, 46, 47]], device='mps:0')


## Step 5 — Bigram baseline (your turn)

Our first actual model. "Bigram" = the prediction for the next character depends on
**only the single current character** — no longer context, no attention. It's deliberately
dumb: the **baseline we'll try to beat** once attention shows up.

### `nn.Module` — the base class for every model
You subclass `nn.Module` and define two methods:
- `__init__` — create the **learnable layers** (store them as `self.` attributes; PyTorch
  auto-discovers any layer/parameter you assign and exposes it via `.parameters()`, which is
  how the optimizer and autograd find them). Call `super().__init__()` first.
- `forward(...)` — the actual computation, run when you call `model(x)`.

### The embedding table *is* the whole model here
`nn.Embedding(num_rows, dim)` is a **learnable lookup table**: row `i` is a length-`dim`
vector attached to token `i`. Feed it integer ids, it returns the matching rows.

The trick for a bigram model: make the table **`(vocab_size, vocab_size)`**. Then looking up
token `i` returns `vocab_size` numbers directly — and we *interpret those as the raw scores
(**logits**) for the next character*. No hidden layers. Token → next-token scores, one lookup.

Shape flow: `idx` is `(B, T)` → embedding → **`(B, T, C)`** with `C = vocab_size`. Each of the
`B*T` positions gets its own length-`C` logits vector. Hold `(B, T, C)` in your head.

### `forward` returns **logits AND loss**
For training we also pass `targets` (shape `(B, T)`) and compute loss with `F.cross_entropy`.
**The gotcha that bites everyone:**

> `F.cross_entropy` wants input `(N, C)` and target `(N,)` — the **class dimension must be
> dim 1**. But your logits are `(B, T, C)` — class dimension is *last*.

So **reshape** before calling it: flatten the `(B, T)` grid into `N = B*T` independent
classifications → logits `(B*T, C)`, targets `(B*T,)`. Why valid? Each position genuinely is
its own "which of C chars comes next?" question. (Also handle `targets=None` → return
`loss=None`, so the same `forward` works for generation, which has no targets.)

### The sanity check that proves it works
At init, weights are random → the model predicts ≈ **uniform** over 65 chars. Cross-entropy of
a uniform guess is `-ln(1/65) = ln(65) ≈ 4.17`. So your first loss should print **~4.1–4.2**.
Wildly off (10, or 0.5) → a reshape or vocab bug. This one number catches a shocking amount.

**This round:** build `__init__` + `forward`. We add `generate` next, once the loss looks right.

In [31]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # TODO(human): create the lookup table.
        #   What shape makes each token map DIRECTLY to logits over the next token?
        #   (both dims are vocab_size — see the writeup)a
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensors of integer ids.
        # 1) look up logits: (B, T) -> (B, T, C)   where C = vocab_size
        logits = self.token_embedding_table(idx)
        B, T, C = logits.shape

        # 2) loss:
        #    - if targets is None -> loss = None  (generation path, no answer key)
        #    - else: F.cross_entropy needs (N, C) and (N,). You have (B, T, C) and (B, T).
        #            Pull B, T, C out of logits.shape, then .view(...) to collapse B and T.
        #            (hint: logits -> (B*T, C), targets -> (B*T,))
        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(B*T,C), targets.view(B*T))
        
        return logits, loss


# --- Harness (given): instantiate, run one forward, check the loss ---
model = BigramLanguageModel(vocab_size).to(device)
logits, loss = model(xb, yb)
print('logits.shape', logits.shape)   # expect (B, T, vocab_size) = (4, 8, 65)
print('loss', loss.item())            # expect ~4.1-4.2  (ln(65) = 4.174)
print('sanity: ln(65) =', torch.log(torch.tensor(float(vocab_size))).item())


logits.shape torch.Size([4, 8, 65])
loss 4.507084846496582
sanity: ln(65) = 4.174387454986572


## Step 5b — Generation (your turn)

The model can score; now make it *write*. `generate` runs the model **autoregressively**:
predict the next char, append it, feed the longer sequence back in, repeat. The model's own
output becomes its next input — that feedback loop is what "generation" means.

### The loop, one step at a time
Given `idx` of shape `(B, T)` (a running context), each step does:

1. **Forward** → `logits, loss = self(idx)`. (No targets, so `loss` is `None` — this is
   exactly why you handled `targets=None`.) `logits` is `(B, T, C)`.
2. **Keep only the last time step.** You want the prediction for the position *after* the
   current end, i.e. `logits[:, -1, :]` → shape `(B, C)`. (A bigram only *uses* the last
   char anyway, but this slicing is how every autoregressive model does it — so build the
   habit now.)
3. **Softmax** those logits into probabilities over the vocab → `probs`, shape `(B, C)`.
4. **Sample** one id per row with `torch.multinomial(probs, num_samples=1)` → `(B, 1)`.
   *Sample*, don't `argmax` — sampling gives variety; argmax would spit the same char forever.
5. **Append** the sampled id onto `idx` along the **time axis** (`dim=1`) with `torch.cat`,
   so `idx` grows `(B, T) → (B, T+1)`. Next iteration sees the longer context.

Repeat `max_new_tokens` times. Return the grown `idx`.

### Why it'll sound like gibberish (and that's correct)
This model is **untrained** — random weights. So it'll emit random characters. That's the
*right* result for now: you're testing the **plumbing** (shapes, the loop, decode), not
quality. After Step 6 (training) the *same* `generate` will start producing word-shaped text.

### Shapes to keep straight
`(B, T, C)` → slice last step → `(B, C)` → softmax → `(B, C)` → multinomial → `(B, 1)` →
cat onto `idx` → `(B, T+1)`. If a shape surprises you, print it inside the loop.

In [ ]:
# Add a `generate` method to the model. Easiest: define it here and attach it to the class,
# OR (cleaner) go back and add it inside `class BigramLanguageModel` and re-run that cell.
# Either way, fill in the TODO(human) below.

def generate(self, idx, max_new_tokens):
    # idx is (B, T): the running context of already-known ids.
    for _ in range(max_new_tokens):
        # 1) forward pass on the current context. (targets=None -> loss is None)
        logits, loss = self(idx)                      # logits: (B, T, C)

        # 2) keep ONLY the last time step's logits -> (B, C)
        logits = logits[:, -1, :]

        # 3) softmax over the vocab (which dim is the class dim?) -> probs (B, C)
        probs = F.softmax(logits, dim=1)

        # 4) sample one id per row -> (B, 1)   (torch.multinomial, NOT argmax)
        idx_next = torch.multinomial(probs, num_samples=1)

        # 5) append along the time axis (dim=1) so idx grows (B, T) -> (B, T+1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

BigramLanguageModel.generate = generate   # attach the method to the class

# --- Smoke test (given): start from a single newline char, generate 300 chars ---
start = torch.zeros((1, 1), dtype=torch.long, device=device)   # (B=1, T=1), id 0 = '\n'
out = model.generate(start, max_new_tokens=300)                # -> (1, 301)
print(decode(out[0].tolist()))



U3VAWktV .nSHzaOWCOpU&Io&nQ! Ncdr,NGPBLa $.rqDUIrZY-jj&irSpG:SBFxBqIoqlAAouTyOoPHDy qRgQeiTaHbnZlqZAVFh:oYtsOoXzAoEWZooHJRLiQp:jmz;xPMWTY'wrhlSmmoLzFIoyqosqLDkHga;GrwvZPHg&!!33aITLa p:Y$dEcfDiSegoSd:xAKKZs$JMuPkuHtVpVOkgol&IokJGx.3AHus$cQsDt!yT,i,oTyg$
roCV;.Slrcx:-$T&mnFBeXjU&hrlTWKCcBVlp,NQpc&qNxf


## Step 6 — Training loop (your turn)

Time to make the loss actually fall. Training is a loop that repeats one ritual:
**sample a batch → compute loss → backprop → nudge the weights.** Thousands of times.

You need two things:

1. **An optimizer.** It holds `model.parameters()` and knows how to update them from their
   gradients. `AdamW` is the standard choice; it takes a learning rate. (Bigram is tiny, so
   you can afford a fairly high `lr` — think `1e-2`-ish, not the `3e-4` you'll want later.)

2. **The loop.** Each iteration: get a fresh training batch, forward to get `loss`, then the
   three-line backward ritual that every PyTorch model uses — *in a specific order*. Figure
   out the order and what each line does; that ordering is the actual lesson here. One of the
   three is easy to forget, and forgetting it silently corrupts training rather than crashing.

Print the loss every few hundred steps so you can watch it drop. Starting near ~4.5, a
trained bigram bottoms out around **~2.4–2.5** — that's the ceiling of "next char from
current char alone," and the wall that attention will let us break through.

Afterward, re-run your `generate` cell. Same function, trained weights → it should go from
pure noise to something with word-ish chunks and spacing. Not good. Just *less random*.

In [35]:
# TODO(human): train the bigram model.

# 1) create the optimizer over model.parameters()
optimizer = torch.optim.AdamW(model.parameters(), 1e-2)

max_iters = 3000
for step in range(max_iters):
    # 2) sample a batch of training data
    xb, yb = get_batch('train')

    # 3) forward -> loss
    logits, loss = model(xb, yb)

    # 4) the three-line backward ritual (correct order matters):
    #      - clear old gradients
    #      - backpropagate this loss
    #      - step the weights
    # ...
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 300 == 0:
        print(f"step {step:4d} | loss {loss.item():.4f}")

print("final loss:", loss.item())


step    0 | loss 4.6073
step  300 | loss 3.1548
step  600 | loss 2.7091
step  900 | loss 2.9482
step 1200 | loss 2.9765
step 1500 | loss 2.1456
step 1800 | loss 2.6716
step 2100 | loss 2.6756
step 2400 | loss 2.5397
step 2700 | loss 2.5590
final loss: 2.2114269733428955


In [40]:
out = model.generate(start, max_new_tokens=500)
print(decode(out[0].tolist()))


Herxcan tf t;
LOLLULOMoundikisthint me t.
Foussmandis, s agore fud tere inond veladon sarnd ithy iswermangatheveche O, s wad ad ly no toweal t you, fenom wisheshakidon hing gin o czKI the VIrepp hasther wenan, galarveainey wans 't s tadidorade byollonoththinodg m
OFoo an her hend:
Tian a be fist carepe 'ds wiavene re enZAno &trit, my, te d taing n;
Hw RShave fuiout, aid row sur arsave taind ss thes tWe.
WUEGUparert thur helove p-
Amiby.
ONG beancixSathize hanthir he wathay ag heveslls tha wikes 
